<a href="https://colab.research.google.com/github/sathushetty7/Bank-fraud-detection-ml/blob/main/bank_fraud_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
# Install the kaggle library
!pip install kaggle

In [24]:
#Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
# The dataset 'mlg-ulb/creditcardfraud' contains a file named 'creditcard.csv'
file_path = "creditcard.csv"

# Load the latest version using the recommended function
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "mlg-ulb/creditcardfraud",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documentation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

In [ ]:
df

In [ ]:
# DATASET INFO
df.info()

In [ ]:
# STATISTICAL SUMMARY
display(df.describe().T)

In [ ]:
#data cleaning


# Missing Values

missing = df.isnull().sum()

display(
    missing[missing > 0].sort_values(ascending=False)
)

if missing.sum() == 0:
    print("No missing values found.")


# Duplicate Rows

duplicates = df.duplicated().sum()

print(f"Duplicate rows: {duplicates:,}")

In [ ]:
# Target Distribution

class_counts = df["Class"].value_counts()

legitimate = class_counts.get(0, 0)
fraud = class_counts.get(1, 0)
total = len(df)

fraud_percentage = (fraud / total) * 100

print(f"Total transactions     : {total:,}")
print(f"Legitimate transactions: {legitimate:,}")
print(f"Fraudulent transactions: {fraud:,}")
print(f"Fraud percentage       : {fraud_percentage:.4f}%")

In [ ]:
# Fraud Distribution Plot

plt.figure(figsize=(7, 5))

sns.countplot(
    data=df,
    x="Class"
)

plt.title("Legitimate vs Fraudulent Transactions")
plt.xlabel("Transaction Class (0 = Legitimate, 1 = Fraud)")
plt.ylabel("Number of Transactions")
plt.xlim(0, 15)
plt.show()



In [ ]:
# Transaction Amount

plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x="Amount",
    bins=100
)

plt.title("Transaction Amount Distribution")
plt.xlabel("Transaction Amount")
plt.ylabel("Frequency")
plt.xlim(0, 10000)
plt.show()





In [ ]:
# Fraud Amount Distribution

plt.figure(figsize=(10, 5))

sns.histplot(
    data=df[df["Class"] == 1],
    x="Amount",
    bins=50
)

plt.title("Fraudulent Transaction Amount Distribution")
plt.xlabel("Transaction Amount")
plt.ylabel("Fraudulent Transactions")

plt.show()



In [ ]:
# Time Distribution

plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x="Time",
    bins=100
)

plt.title("Transaction Distribution Over Time")
plt.xlabel("Time")
plt.ylabel("Transactions")

plt.show()




In [ ]:
# Fraud Over Time

plt.figure(figsize=(10, 5))

sns.histplot(
    data=df[df["Class"] == 1],
    x="Time",
    bins=50
)

plt.title("Fraudulent Transactions Over Time")
plt.xlabel("Time")
plt.ylabel("Fraudulent Transactions")

plt.show()

In [ ]:
#  Correlation Matrix

plt.figure(figsize=(16, 12))

correlation = df.corr(numeric_only=True)

sns.heatmap(
    correlation,
    cmap="coolwarm",
    center=0,
    linewidths=0.1
)

plt.title("Feature Correlation Matrix")

plt.show()


In [ ]:
#  Correlation With Fraud

fraud_correlation = (
    correlation["Class"]
    .sort_values(ascending=False)
)

display(fraud_correlation.to_frame(name="Correlation"))



In [ ]:
# Fraud vs Legitimate Amount

amount_stats = df.groupby("Class")["Amount"].agg(
    ["count", "mean", "median", "min", "max"]
)

amount_stats.index = [
    "Legitimate",
    "Fraud"
]

display(amount_stats)


In [ ]:

# Final Dataset Summary

print(f"Total transactions : {total:,}")
print(f"Legitimate         : {legitimate:,}")
print(f"Fraud              : {fraud:,}")
print(f"Fraud percentage   : {fraud_percentage:.4f}%")
print(f"Missing values     : {df.isnull().sum().sum():,}")
print(f"Duplicate rows     : {duplicates:,}")
print("========================================")

In [ ]:

# DATA PREPARATION + FEATURE ENGINEERING

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

# Create a copy of the dataset


data = df.copy()

print("Dataset shape:", data.shape)



# Separate features (X) and target (y)

X = data.drop("Class", axis=1)
y = data["Class"]

print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)



# Scale Time and Amount
# V1-V28 are already PCA-transformed.
# Time and Amount are on different scales, so we standardize them.

scaler = StandardScaler()

X[["Time", "Amount"]] = scaler.fit_transform(
    X[["Time", "Amount"]]
)

print("\nTime and Amount have been standardized.")


# Train/Test Split

# stratify=y keeps the fraud/legitimate ratio similar
# in both training and testing datasets.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)



# Check the resulting shapes


print("\n===== DATA SPLIT =====")

print("Training features :", X_train.shape)
print("Testing features  :", X_test.shape)
print("Training target   :", y_train.shape)
print("Testing target    :", y_test.shape)



#  Check fraud distribution


train_counts = y_train.value_counts()

print("Legitimate:", train_counts[0])
print("Fraud     :", train_counts[1])
print("Fraud %   :", round((train_counts[1] / len(y_train)) * 100, 4))




test_counts = y_test.value_counts()

print("Legitimate:", test_counts[0])
print("Fraud     :", test_counts[1])
print("Fraud %   :", round((test_counts[1] / len(y_test)) * 100, 4))



#  Verify no missing values



print("X_train:", X_train.isnull().sum().sum())
print("X_test :", X_test.isnull().sum().sum())



# Verify target values

print("Training:", sorted(y_train.unique()))
print("Testing :", sorted(y_test.unique()))



# Final summary




print("✓ Features and target separated")
print("✓ Time and Amount standardized")
print("✓ Stratified train/test split completed")
print("✓ Fraud distribution verified")
print("✓ Missing values checked")
print("✓ Data is ready for model training")

In [ ]:

# BASELINE MODELS
# Logistic Regression + Random Forest



from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import pandas as pd
import matplotlib.pyplot as plt



# Evaluation function


def evaluate_model(model, name):

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC": average_precision_score(y_test, y_prob)
    }

    print(f"\n{name}")
    print("-" * 40)

    for metric, value in results.items():
        if metric != "Model":
            print(f"{metric:<10}: {value:.4f}")

    cm = confusion_matrix(y_test, y_pred)

    ConfusionMatrixDisplay(
        cm,
        display_labels=["Legitimate", "Fraud"]
    ).plot()

    plt.title(f"{name} — Confusion Matrix")
    plt.show()

    return results


# Logistic Regression


logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_results = evaluate_model(
    logistic_model,
    "Logistic Regression"
)



# Random Forest

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

random_forest_results = evaluate_model(
    random_forest_model,
    "Random Forest"
)



#  Compare models

baseline_results = pd.DataFrame([
    logistic_results,
    random_forest_results
])

print("\n===== BASELINE MODEL COMPARISON =====")

display(
    baseline_results.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
        "ROC-AUC": "{:.4f}",
        "PR-AUC": "{:.4f}"
    })
)

In [ ]:
#  IMBALANCE HANDLING + XGBOOST

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import pandas as pd
import matplotlib.pyplot as plt